In [1]:
# read rscv results for regression hpt
import numpy as np
import pandas as pd 
import os
import re
from pathlib import Path

rscv_dir = Path(r"C:\Users\A S U S\Documents\Study\ClimateData\CD-Workspace\Data\Processed\Bias_correction_daily\Regressor_model\rscv_results")

In [2]:
def compile_rscv_results(rscv_dir):
    results_list = []

    # Regex to extract: Group 1 (cluster), Group 2 (season), Group 3 (model type)
    pattern = re.compile(r"^([^_]+)_(.*)_(rf|xgb)\.csv$")

    for filename in os.listdir(rscv_dir):
        if filename.endswith(".csv"):
            match = pattern.match(filename)
            if match:
                cluster_str = match.group(1)
                season_str = match.group(2)
                model_type = match.group(3)
                
                # Format cluster: convert '9.0' to 'All'
                cluster_val = "All" if cluster_str == "9.0" else cluster_str
                
                # Format season: convert 'allSeasons' to 'All'
                season_val = "All" if season_str == "allSeasons" else season_str
                
                filepath = os.path.join(rscv_dir, filename)
                
                try:
                    # Read the specific rscv CSV
                    df = pd.read_csv(filepath)
                    
                    # Filter for the best performing model(s)
                    best_model_rows = df[df['rank_test_score'] == 1]
                    
                    # If there's a tie for 1st place, grab the first one with .iloc[0]
                    best_n_estimators = best_model_rows['param_regressor__n_estimators'].iloc[0]
                    
                    results_list.append({
                        'cluster': cluster_val,
                        'season': season_val,
                        'model type': model_type.upper(),
                        'n_estimators': int(best_n_estimators) # Ensure it reads as an integer
                    })
                except Exception as e:
                    print(f"Error reading {filename}: {e}")

    # Compile into DataFrame
    summary_df = pd.DataFrame(results_list)

    # Sort for readability
    if not summary_df.empty:
        summary_df = summary_df.sort_values(by=['cluster', 'season', 'model type']).reset_index(drop=True)
        
    return summary_df

# Run the function and display the table
rscv_summary_table = compile_rscv_results(rscv_dir)

print("\n--- RSCV Best n_estimators ---")
print(rscv_summary_table.to_string())


--- RSCV Best n_estimators ---
   cluster        season model type  n_estimators
0      0.0           All         RF          5000
1      0.0           All        XGB           100
2      0.0       monsoon         RF         15000
3      0.0       monsoon        XGB          1000
4      0.0  post_monsoon         RF           100
5      0.0  post_monsoon        XGB           500
6      0.0   pre_monsoon         RF          3000
7      0.0   pre_monsoon        XGB          1000
8      0.0        winter         RF          5000
9      0.0        winter        XGB          1000
10     1.0           All         RF          1000
11     1.0           All        XGB          1000
12     1.0       monsoon         RF         15000
13     1.0       monsoon        XGB           500
14     1.0  post_monsoon         RF          1000
15     1.0  post_monsoon        XGB           500
16     1.0   pre_monsoon         RF          7000
17     1.0   pre_monsoon        XGB          1000
18     1.0        